# MySGD: hand-rolled optimizer

Based on Lecture 3 (SGD Optimizer & LR Scheduler). Implements SGD with optional weight decay and gradient clipping, mirroring torch.optim.SGD at a minimal level.

In [5]:
import torch
from torch import nn, Tensor
from typing import Iterable, Optional

# keep runs deterministic for demos
torch.manual_seed(0)


In [ ]:
class MySGD:
    """Minimal SGD optimizer with optional weight decay and gradient clipping.

    Public API mirrors torch-style optimizers:
      * zero_grad(): clears stored gradients in-place
      * step(): (optional) clip gradients, apply weight decay, then SGD update
    """
    def __init__(self, params: Iterable[Tensor], lr: float = 0.1, weight_decay: float = 0.0, grad_clip: Optional[float] = None):
        # Keep only trainable tensors (requires_grad=True) and store as a list for reuse
        self.params = [p for p in params if p.requires_grad]
        if not self.params:
            raise ValueError('No trainable parameters provided to MySGD.')
        self.lr = lr
        self.weight_decay = weight_decay
        self.grad_clip = grad_clip

    def zero_grad(self):
        # Manually clear gradients so the next backward() starts fresh
        for p in self.params:
            # p.grad = None
            if p.grad is not None:
                p.grad.zero_()

    def _clip_grads(self):
        # Global norm clipping: rescale gradients if their combined norm is too large
        if self.grad_clip is None:
            return
        
        grads = [p.grad for p in self.params if p.grad is not None]
        # assert p.grad is not None

        if not grads:
            return
        total_norm = torch.norm(torch.stack([g.norm() for g in grads]))
        if total_norm > self.grad_clip:
            scale = self.grad_clip / (total_norm + 1e-8)
            for g in grads:
                g.mul_(scale)

    # clip by value
    # torch.clamp(grads, 0, self.grad_clip)



    @torch.no_grad()
    def step(self):
        # Optionally clip first, then apply weight decay + SGD update for each parameter
        self._clip_grads()
        for p in self.params:
            if p.grad is None:
                continue  # nothing to update if no gradient accumulated
            grad = p.grad
            if self.weight_decay != 0.0:
                grad = grad + self.weight_decay * p.data  # coupled weight decay
            # p.data.add_(grad, alpha=-self.lr)  # in-place: p = p - lr * grad
            p.data = p.data - self.lr * grad

## Demo: fit a noisy 1D linear regression
We train a single `nn.Linear` using `MySGD` to show weight decay + grad clipping in action.

In [7]:
# synthetic data y = 2x - 0.5 + noise
N = 256
X = torch.randn(N, 1)
true_w, true_b = 2.0, -0.5
y = true_w * X + true_b + 0.1 * torch.randn(N, 1)

# Small linear model; gradients flow into weight and bias parameters
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = MySGD(model.parameters(), lr=0.1, weight_decay=1e-2, grad_clip=1.0)

loss_hist = []
for epoch in range(50):
    optimizer.zero_grad()  # clear stale grads
    pred = model(X)
    loss = criterion(pred, y)
    loss.backward()       # populate .grad on parameters
    optimizer.step()      # apply SGD update
    loss_hist.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:02d} | loss={loss.item():.4f} | lr={optimizer.lr}')

print('True w,b:', true_w, true_b)
print('Learned w,b:', model.weight.item(), model.bias.item())


Epoch 10 | loss=1.6706 | lr=0.1
Epoch 20 | loss=0.1521 | lr=0.1
Epoch 30 | loss=0.0157 | lr=0.1
Epoch 40 | loss=0.0120 | lr=0.1
Epoch 50 | loss=0.0118 | lr=0.1
True w,b: 2.0 -0.5
Learned w,b: 1.9792578220367432 -0.48698362708091736


## Observations
- Weight decay keeps parameters small while converging.
- Gradient clipping prevents large updates if the norm spikes (try increasing lr to see its effect).
- `MySGD` exposes a simple API compatible with the Lecture 3 optimizer discussion.